##**Code to Create the StateBridge Class**

In [1]:
import asyncio
from dataclasses import dataclass, field
from typing import List, Optional

@dataclass
class VLMCommand:
    """The movement command structure passed between loops."""
    waypoints: List[List[float]]
    obstacles: List[str]
    confidence: float
    raw_response: str
    action: str = "MOVE"

class StateBridge:
    def __init__(self):
        # Create a 1-slot asynchronous queue. maxsize=1 ensures it only holds the newest command.
        self.queue = asyncio.Queue(maxsize=1)
        # An event flag to instantly signal the IK loop when a fresh command arrives
        self.new_command_event = asyncio.Event()

    def has_command(self) -> bool:
        """Returns True if there is an item currently sitting in the queue."""
        return not self.queue.empty()

    async def put_command(self, cmd: VLMCommand):
        """
        Puts a new command into the bridge. If an old one is already there,
        it clears it out first so the robot never acts on old, stale data.
        """
        # If the queue is full (has 1 item), clear it out immediately
        if self.queue.full():
            try:
                self.queue.get_nowait()  # Discard the old, stale command
            except asyncio.QueueEmpty:
                pass

        # Put the new command in
        await self.queue.put(cmd)
        # Trigger the event flag to alert the downstream hardware loop
        self.new_command_event.set()

    async def get_latest_command(self) -> Optional[VLMCommand]:
        """
        Pulls the latest command out of the bridge.
        Clears the alert flag since the command has been read.
        """
        if self.queue.empty():
            return None

        # Pull the item out
        cmd = await self.queue.get()
        # Reset the event flag because we just consumed the newest data
        self.new_command_event.clear()
        return cmd

print("🚀 Stage 1 Complete: StateBridge class successfully compiled in your new notebook!")

🚀 Stage 1 Complete: StateBridge class successfully compiled in your new notebook!


##**Test Code**

In [2]:
import asyncio

async def run_adversarial_test():
    # 1. Instantiate our fresh StateBridge
    bridge = StateBridge()

    print("🚀 Initiating Stage 2 Test: Pushing 5 commands in rapid succession...")
    print("----------------------------------------------------------------")

    # 2. Loop to rapidly fire 5 commands into the single-slot bridge
    for i in range(1, 6):
        # Create a mock command with a changing coordinate value to track it
        mock_cmd = VLMCommand(
            waypoints=[[0.1 * i, 0.2 * i, 0.3 * i]],
            obstacles=[f"obstacle_id_{i}"],
            confidence=0.90 + (i * 0.01),
            raw_response=f"Generated tracking frame version {i}"
        )

        print(f"📥 Action: Vision Loop puts Command #{i} (Waypoint X = {0.1 * i:.2f}m)")

        # Drop it into the bridge
        await bridge.put_command(mock_cmd)

        # 3. Peek inside to verify the queue length never climbs past 1
        current_size = bridge.queue.qsize()
        print(f"📊 Status: Current Queue Size = {current_size} slot(s) full.")

    print("----------------------------------------------------------------")
    print("🔍 Fetching final remaining command from the bridge mailbox...")

    # 4. Pull out what is left in the queue
    final_cmd = await bridge.get_latest_command()

    if final_cmd:
        print(f"✅ VERIFICATION COMPLETE!")
        print(f"• Retrieved Command Raw Response: '{final_cmd.raw_response}'")
        print(f"• Retrieved Command Waypoint X: {final_cmd.waypoints[0][0]:.2f}m")

        # If it matches Command #5, our drop-stale architecture is working flawlessly
        if "version 5" in final_cmd.raw_response:
            print("\n🎉 SUCCESS: The bridge successfully dropped all 4 stale commands automatically! Only the 5th (latest) command remained in the queue.")
        else:
            print("\n❌ FAILURE: The bridge did not correctly prioritize the latest item.")
    else:
        print("\n❌ FAILURE: Queue is completely empty.")

# Execute the test suite inside the notebook's running event loop
await run_adversarial_test()

🚀 Initiating Stage 2 Test: Pushing 5 commands in rapid succession...
----------------------------------------------------------------
📥 Action: Vision Loop puts Command #1 (Waypoint X = 0.10m)
📊 Status: Current Queue Size = 1 slot(s) full.
📥 Action: Vision Loop puts Command #2 (Waypoint X = 0.20m)
📊 Status: Current Queue Size = 1 slot(s) full.
📥 Action: Vision Loop puts Command #3 (Waypoint X = 0.30m)
📊 Status: Current Queue Size = 1 slot(s) full.
📥 Action: Vision Loop puts Command #4 (Waypoint X = 0.40m)
📊 Status: Current Queue Size = 1 slot(s) full.
📥 Action: Vision Loop puts Command #5 (Waypoint X = 0.50m)
📊 Status: Current Queue Size = 1 slot(s) full.
----------------------------------------------------------------
🔍 Fetching final remaining command from the bridge mailbox...
✅ VERIFICATION COMPLETE!
• Retrieved Command Raw Response: 'Generated tracking frame version 5'
• Retrieved Command Waypoint X: 0.50m

🎉 SUCCESS: The bridge successfully dropped all 4 stale commands automatica

In [3]:
import asyncio

async def run_remaining_adversarial_tests():
    bridge = StateBridge()
    print("🚀 Initiating Remaining Adversarial Tests...")
    print("----------------------------------------------------------------")

    # === TEST 1: EMPTY QUEUE GETS ===
    print("🔍 Scenario 1: Attempting to read from a completely empty bridge...")
    # The bridge should return None immediately instead of blocking/hanging
    fetched_cmd = await bridge.get_latest_command()

    print(f"📊 Status: Result of empty get = {fetched_cmd}")
    if fetched_cmd is None:
        print("✅ SUCCESS: Empty queue get returned None instantly without blocking.")
    else:
        print("❌ FAILURE: Unexpected item found in empty queue.")

    print("----------------------------------------------------------------")

    # === TEST 2: CONCURRENT GETS ===
    print("🔍 Scenario 2: Launching 3 reader tasks to grab 1 command at the same instant...")

    # Put exactly ONE command into the bridge
    test_cmd = VLMCommand(
        waypoints=[[0.6, 0.6, 0.6]],
        obstacles=["safety_gate"],
        confidence=0.99,
        raw_response="Concurrent test frame"
    )
    await bridge.put_command(test_cmd)

    # Define a helper reader function for the concurrent test
    async def reader_worker(worker_id: int):
        cmd = await bridge.get_latest_command()
        return worker_id, cmd

    # Fire off 3 reader tasks simultaneously
    reader_tasks = [reader_worker(id) for id in range(1, 4)]
    results = await asyncio.gather(*reader_tasks)

    # Check who won the race and who got empty data
    for worker_id, cmd in results:
        if cmd is not None:
            print(f"🥇 Task #{worker_id} WON the race! Successfully retrieved: '{cmd.raw_response}'")
        else:
            print(f"🥈 Task #{worker_id} received None (Queue was already safely emptied)")

    print("----------------------------------------------------------------")
    print("🎉 ALL ADVERSARIAL TESTING COMPLETE!")

# Execute the tests
await run_remaining_adversarial_tests()

🚀 Initiating Remaining Adversarial Tests...
----------------------------------------------------------------
🔍 Scenario 1: Attempting to read from a completely empty bridge...
📊 Status: Result of empty get = None
✅ SUCCESS: Empty queue get returned None instantly without blocking.
----------------------------------------------------------------
🔍 Scenario 2: Launching 3 reader tasks to grab 1 command at the same instant...
🥇 Task #1 WON the race! Successfully retrieved: 'Concurrent test frame'
🥈 Task #2 received None (Queue was already safely emptied)
🥈 Task #3 received None (Queue was already safely emptied)
----------------------------------------------------------------
🎉 ALL ADVERSARIAL TESTING COMPLETE!
